In [1]:
import requests
import pandas as pd
import re
import time

def scrape_olx_jakarta_full(target_data=43000):
    all_data = []
    seen_urls = set()
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

    price_brackets = [
        0, 30000000, 50000000, 75000000, 100000000, 125000000, 150000000,
        175000000, 200000000, 250000000, 300000000, 400000000, 500000000,
        750000000, 1000000000, 1500000000, 2000000000, 5000000000
    ]

    sort_options = ["", "desc-creation", "asc-price", "desc-price"]

    print("=======================================================")
    print("=== SCRAPING FULL DATASET MOBIL BEKAS DKI JAKARTA ===")
    print("=======================================================\n")

    for b in range(len(price_brackets) - 1):
        min_p = price_brackets[b]
        max_p = price_brackets[b + 1]

        print(f"\n--- Memproses Rentang Harga: Rp {min_p:,} s/d Rp {max_p:,} ---")

        for sort_type in sort_options:
            page = 0

            while page < 25:
                url = (
                    f"https://www.olx.co.id/api/relevance/v2/search?"
                    f"category=198&"
                    f"location=2000007&"
                    f"location_facet=2000007&"
                    f"filter_price_between={min_p}%3A{max_p}&"
                    f"page={page}"
                )
                if sort_type != "":
                    url += f"&sorting={sort_type}"

                try:
                    res = requests.get(url, headers=headers, timeout=10)
                    if res.status_code != 200:
                        break
                    parsed = res.json()
                except Exception:
                    break

                items = parsed.get("data", [])
                if not items:
                    break

                for item in items:
                    judul_str = str(item.get("title")) if item.get("title") is not None else None

                    merk = None
                    model_mobil = None
                    tahun = None
                    jarak_tempuh = None
                    bahan_bakar = None
                    transmisi = None
                    tipe_penjual = None
                    kapasitas_mesin = None
                    warna = None
                    tipe_bodi = None

                    # 1. Parsing dari Object 'parameters'
                    parameters = item.get("parameters") or []
                    for p in parameters:
                        key_str = f"{p.get('key', '')} {p.get('formatted_key', '')} {p.get('key_name', '')} {p.get('id', '')}".lower()
                        val = p.get("value_name") if p.get("value_name") is not None else p.get("formatted_value")
                        if not val:
                            continue
                        val = str(val)

                        if re.search(r"brand|merk|make", key_str):
                            merk = val
                        elif re.search(r"model", key_str) and not re.search(r"brand|merk", key_str):
                            model_mobil = val
                        elif re.search(r"year|tahun", key_str):
                            tahun = val
                        elif re.search(r"mileage|jarak|km", key_str):
                            jarak_tempuh = val
                        elif re.search(r"fuel|bahan_bakar|bensin", key_str):
                            bahan_bakar = val
                        elif re.search(r"transmis", key_str):
                            transmisi = val
                        elif re.search(r"seller|penjual", key_str):
                            tipe_penjual = val
                        elif re.search(r"capacity|kapasitas|cc|engine", key_str):
                            kapasitas_mesin = val
                        elif re.search(r"color|warna", key_str):
                            warna = val
                        elif re.search(r"body|bodi", key_str):
                            tipe_bodi = val

                    # 2. Fallback Parsing dari Judul
                    if judul_str:
                        words = judul_str.strip().split()
                        if not merk and len(words) > 0:
                            merk = words[0]
                        if not model_mobil:
                            model_mobil = words[1] if len(words) >= 2 else (words[0] if len(words) > 0 else None)

                        if not tahun:
                            match_yr = re.search(r"\b(19|20)\d{2}\b", judul_str)
                            if match_yr:
                                tahun = match_yr.group(0)

                        if not transmisi:
                            if re.search(r"AT|Automatic|Matic|CVT", judul_str, re.IGNORECASE):
                                transmisi = "Automatic"
                            elif re.search(r"MT|Manual", judul_str, re.IGNORECASE):
                                transmisi = "Manual"

                    # 3. Nama Bursa / Showroom / Toko Penjual
                    bursa_mobil = "Penjual Perorangan"
                    store_name = (item.get("store") or {}).get("name")
                    user_name = (item.get("user") or {}).get("name")

                    if store_name:
                        bursa_mobil = str(store_name)
                    elif user_name:
                        bursa_mobil = str(user_name)

                    # 4. Lokasi
                    locs = item.get("locations_resolved") or {}
                    prov = locs.get("ADMIN_LEVEL_1_name") or "Jakarta D.K.I."
                    kota = locs.get("ADMIN_LEVEL_2_name") or locs.get("ADMIN_LEVEL_3_name") or prov
                    kecamatan = locs.get("ADMIN_LEVEL_3_name")

                    # 5. Harga Display & Numerik
                    price_obj = (item.get("price") or {}).get("value") or {}
                    harga_str = price_obj.get("display") if price_obj.get("display") is not None else price_obj.get("raw")
                    harga_num = None
                    if harga_str is not None:
                        digits = re.sub(r"[^0-9]", "", str(harga_str))
                        if digits:
                            harga_num = float(digits)

                    # 6. URL Detail & Deduplikasi Real-Time
                    item_id = item.get("id")
                    url_detail = f"https://www.olx.co.id/item/{item_id}" if item_id else None

                    if url_detail and url_detail in seen_urls:
                        continue
                    if url_detail:
                        seen_urls.add(url_detail)

                    row = {
                        "Judul": judul_str,
                        "Kategori": "Mobil Bekas",
                        "Provinsi": prov,
                        "Kota_Kabupaten": kota,
                        "Kecamatan": kecamatan,
                        "Harga_Teks": str(harga_str) if harga_str is not None else None,
                        "Harga_IDR": harga_num,
                        "Merk": merk,
                        "Model": model_mobil,
                        "Tahun": tahun,
                        "Jarak_Tempuh": jarak_tempuh,
                        "Tipe_Bahan_Bakar": bahan_bakar,
                        "Transmisi": transmisi,
                        "Tipe_Penjual": tipe_penjual,
                        "Nama_Bursa_Mobil": bursa_mobil,
                        "Kapasitas_Mesin": kapasitas_mesin,
                        "Warna": warna,
                        "Tipe_Bodi": tipe_bodi,
                        "URL_Detail": url_detail
                    }
                    all_data.append(row)

                    print(f"Proses... | Total Data Unik Terkumpul: {len(all_data)} / {target_data}", end="\r")
                    if len(all_data) >= target_data:
                        break

                if len(all_data) >= target_data:
                    break

                page += 1
                time.sleep(0.2)

            if len(all_data) >= target_data:
                break

        if len(all_data) >= target_data:
            break

    df = pd.DataFrame(all_data)
    return df

# ==============================================================================
# EKSEKUSI SCRAPING & SIMPAN KE EXCEL
# ==============================================================================
if __name__ == "__main__":
    df_jakarta_full = scrape_olx_jakarta_full(target_data=43000)
    df_jakarta_full.to_excel("PSD_DATA_MOBIL_JAKARTA_FULL.xlsx", index=False)
    print(f"\n✅ BERHASIL! {len(df_jakarta_full)} data unik disimpan di 'PSD_DATA_MOBIL_JAKARTA_FULL.xlsx'")

=== SCRAPING FULL DATASET MOBIL BEKAS DKI JAKARTA ===


--- Memproses Rentang Harga: Rp 0 s/d Rp 30,000,000 ---

--- Memproses Rentang Harga: Rp 30,000,000 s/d Rp 50,000,000 ---

--- Memproses Rentang Harga: Rp 50,000,000 s/d Rp 75,000,000 ---

--- Memproses Rentang Harga: Rp 75,000,000 s/d Rp 100,000,000 ---

--- Memproses Rentang Harga: Rp 100,000,000 s/d Rp 125,000,000 ---

--- Memproses Rentang Harga: Rp 125,000,000 s/d Rp 150,000,000 ---

--- Memproses Rentang Harga: Rp 150,000,000 s/d Rp 175,000,000 ---

--- Memproses Rentang Harga: Rp 175,000,000 s/d Rp 200,000,000 ---

--- Memproses Rentang Harga: Rp 200,000,000 s/d Rp 250,000,000 ---

--- Memproses Rentang Harga: Rp 250,000,000 s/d Rp 300,000,000 ---

--- Memproses Rentang Harga: Rp 300,000,000 s/d Rp 400,000,000 ---

--- Memproses Rentang Harga: Rp 400,000,000 s/d Rp 500,000,000 ---

--- Memproses Rentang Harga: Rp 500,000,000 s/d Rp 750,000,000 ---

--- Memproses Rentang Harga: Rp 750,000,000 s/d Rp 1,000,000,000 ---

--- Me